# Stage 4, tier 2: LucaOne as a frozen baseline, and IsoFormer's GTEx table at three seeds

Research plan **Priority 2** (Gantt 41) and the baseline that is not on her list, **IsoFormer
on GTEx** (Gantt 44). Both ask the same question from opposite directions: is one model that
already knows about several modalities better than three models fused afterwards?

* **LucaOne** is 1.8B parameters trained from scratch on a single 39-token vocabulary
  covering nucleotides and amino acids, so whatever relates DNA, RNA and protein is learned
  in pretraining. The plan asks for it as a frozen baseline on **mRNA stability and GTEx**,
  compared against the best single encoder and the best fused model on the same folds, with
  its computational cost and input truncation reported.
* **IsoFormer** fuses three separately pretrained encoders with cross-attention and no codon
  grid. Stage 3 step 4 ran its modality ablation on the GTEx pilot at one seed; this notebook
  adds seeds 1 and 2 so the table carries a spread rather than a point.

**What Stage 3 already settled, and this notebook re-measures rather than assumes.** LucaOne's
published context is 1,280 tokens and its tokenization is character level, one token per
nucleotide, so GTEx's 6,000-nucleotide DNA windows cannot be read at all. The honest scope is
**stability only**, and the first cells below recompute that from the data before anything is
downloaded.

**What is compared, and why it is a probe and not the TextCNN.** LucaOne returns one pooled
vector per sequence. The plan asks for "the same train/test folds and prediction head or
clearly matched probe setting", so the comparison here is a ridge probe on frozen pooled
vectors, run identically for LucaOne and for BioLangFusion's three encoders, on the
benchmark's own published split. The Phase A numbers from the trained TextCNN are printed
beside it for context, clearly marked as a different head.

---

## Setup

In [ ]:
# Standard library
import json
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders, lucaone, sequences
from shared_code.fusion import blf, isoformer, train
from shared_code.fusion import token_embeddings as tokens

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| Apple GPU:", torch.backends.mps.is_available())

### Settings

LucaOne's weights are about 7 GB in float32 and are fetched from the hub on first use, into
the usual Hugging Face cache. On a CUDA GPU the model is loaded in float16, which halves it
to roughly 3.6 GB and is what lets it sit on an 8 GB card beside its activations; on CPU it
stays float32, where float16 arithmetic is slower rather than faster.

`RUN_LUCAONE` is deliberately off. Cell by cell this notebook first measures what one
sequence costs on this machine and prints the projected total; turn the flag on only after
reading that number.

In [ ]:
BENCHMARK = "mrna_stability"          # the only BLF benchmark LucaOne can read whole
CHECKPOINT = "default"                # the joint gene-and-protein model
CACHE_ROOT = Path("../stage3/embeddings")          # BioLangFusion's token caches
LUCAONE_DIR = Path("../stage3/embeddings") / BENCHMARK / "lucaone"
RESULTS_DIR = Path("results") / "lucaone"
GTEX_RESULTS = Path("results") / "gtex_isoformer"

RUN_LUCAONE = False                   # turn on after reading the cost projection below
PROBE_SEEDS = (0, 1, 2)               # fold assignments for the grouped robustness check
BOOTSTRAP_DRAWS = 4000

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE == "cuda" else None
print(f"device {DEVICE}, LucaOne dtype {DTYPE or 'float32'}")
print(f"checkpoints published: {json.dumps(lucaone.CHECKPOINTS, indent=2)}")

## Part 1: what LucaOne can read

The input limit decides the scope before any compute is spent, so it is measured here from
the data rather than recalled from Stage 3. Three modes are possible: the nucleotide stream
alone, the translated protein alone, and the joint input that carries both in one sequence,
which is the mode that makes LucaOne distinctive and the one the context limits hardest.

In [ ]:
coverage = []
for key in datasets.BLF_BENCHMARKS:
    rows = datasets.blf_benchmark_rows(key)
    coverage.append({"benchmark": key, **lucaone.coverage(rows["length"].to_numpy())})

gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)
for column in ("DNA", "RNA", "CDS"):
    length = gtex[column].astype(str).str.strip().str.len()
    coverage.append({"benchmark": f"gtex_pilot/{column}", **lucaone.coverage(length.to_numpy())})

table = pd.DataFrame(coverage).set_index("benchmark")
display(table.style.format({"gene stream": "{:.1%}", "protein stream": "{:.1%}",
                            "joint input": "{:.1%}"}))
print(f"The joint mode needs 4L/3 + 3 tokens, so it caps a coding sequence at "
      f"{lucaone.longest_joint_cds()} nt against LucaOne's {lucaone.CONTEXT_TOKENS}-token context.")

**Verdict on the scope.** The BioLangFusion benchmarks fit one stream at a time, because the
999-nucleotide preprocessing rule leaves room inside 1,280 tokens. GTEx does not: its DNA
track is a fixed 6,000-nucleotide window, nearly five times the context, so a LucaOne GTEx
baseline would have to drop the DNA modality and discard most of the transcripts, which is a
different experiment rather than a baseline. **Gantt 41 therefore runs on mRNA stability and
is closed on GTEx**, and that is a finding worth one line in the write-up: joint pretraining
removes the alignment problem and replaces it with a context ceiling.

One caveat recorded rather than hidden: the checkpoint's own `config.json` reports
`max_position_embeddings` 4096 with `no_position_embeddings` true, so longer inputs may well
run. The published limit is what is used here, because that is the number the paper supports.

## Part 2: the cost of the run, measured before it is started

The plan asks for computational cost to be reported, and a 1.8B-parameter model on a
benchmark of 23,929 sequences is where that matters. This times a handful of sequences and
projects the rest. Nothing is cached from this cell: it is a measurement, not the run.

In [ ]:
rows = datasets.blf_benchmark_rows(BENCHMARK)
print(f"{BENCHMARK}: {len(rows):,} rows, {rows['length'].min()} to {rows['length'].max()} nt")

# LucaOne's gene mode maps A, C, G and both T and U onto the same four tokens, so the
# benchmark's own DNA letters are fed as they are.
PROBE_ROWS = 8
sample = rows["sequence"].head(PROBE_ROWS).tolist()
started = time.perf_counter()
tokenizer, model = lucaone.load(CHECKPOINT, DEVICE, DTYPE)
load_seconds = time.perf_counter() - started
parameters = sum(p.numel() for p in model.parameters())

started = time.perf_counter()
for text in sample:
    lucaone.pooled(text, "gene", tokenizer, model, DEVICE)
per_sequence = (time.perf_counter() - started) / len(sample)
del model, tokenizer
encoders.free_memory()

print(f"LucaOne {CHECKPOINT}: {parameters / 1e9:.2f}B parameters, loaded in {load_seconds:.0f} s")
print(f"{per_sequence:.2f} s per sequence on {DEVICE}")
for streams, label in ((1, "gene stream only"), (2, "gene and protein streams")):
    hours = per_sequence * len(rows) * streams / 3600
    print(f"  {label}: {hours:.1f} hours for {len(rows) * streams:,} forward passes")

### Embedding the benchmark

Two streams, each cached in its own file: the coding sequence read as a gene, and its
translation read as a protein. The cache records the checkpoint, the stream and a fingerprint
of the inputs in order, so a matrix embedded for one stream can never be read back as the
other.

The protein stream uses the grid-aligned translation (`token_embeddings.grid_protein`), the
same string BioLangFusion's protein encoder reads in Phase A, so the two systems are given
the same protein.

In [ ]:
gene_texts = rows["sequence"].tolist()
protein_texts = [tokens.grid_protein(s) for s in rows["sequence"]]
LUCAONE_DIR.mkdir(parents=True, exist_ok=True)

streams = {}
if RUN_LUCAONE:
    for name, texts in (("gene", gene_texts), ("prot", protein_texts)):
        started = time.perf_counter()
        matrix, reused = lucaone.embed_with_cache(
            texts, name, LUCAONE_DIR / f"{CHECKPOINT}_{name}.npz", device=DEVICE,
            checkpoint=CHECKPOINT, dtype=DTYPE)
        streams[name] = matrix
        print(f"  {name}: {matrix.shape}, {'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}")
    lucaone.write_manifest(LUCAONE_DIR / "manifest.json", benchmark=BENCHMARK,
                           checkpoint=lucaone.CHECKPOINTS[CHECKPOINT], rows=len(rows),
                           device=DEVICE, dtype=str(DTYPE), context=lucaone.CONTEXT_TOKENS)
else:
    print("RUN_LUCAONE is off. Read the projection above, then set it to True and rerun.")

## Part 3: the matched comparison

Every model in the table below is a frozen encoder, mean-pooled to one vector per sequence,
read by the same ridge probe on the benchmark's own published split. That is what makes the
comparison fair: the only thing that changes between rows is the representation.

BioLangFusion's three encoders are pooled from the token caches Stage 3 already wrote, so
they are the same weights reading the same rows, with no second extraction. The composition
baseline (GC, GC3, the 64 codon frequencies, log length) is the floor any representation has
to clear to be carrying more than nucleotide counts.

In [ ]:
split = rows["Split"].to_numpy()
y = rows["Value"].to_numpy(dtype=float)
train_rows = np.flatnonzero(split == "train")
test_rows = np.flatnonzero(split == "test")
print(f"published split: {len(train_rows):,} train, {len(test_rows):,} test")

# BioLangFusion's own three encoders, pooled from the Stage 3 token caches.
pooled = {}
for modality, encoder_key in blf.ENCODER_OF_MODALITY.items():
    cache = tokens.summarize(CACHE_ROOT / BENCHMARK / encoder_key)
    pooled[encoders.ENCODERS[encoder_key].label] = tokens.pooled_matrix(cache)
    print(f"  {encoder_key}: {pooled[encoders.ENCODERS[encoder_key].label].shape}")

# The composition floor.
composition = np.array([sequences.composition_features(s) for s in rows["sequence"]])
print(f"  composition features: {composition.shape}")

In [ ]:
def probe_on_split(X, y, train_rows, test_rows):
    # Ridge on the published split: the per-row test predictions and the test score.
    from sklearn.linear_model import RidgeCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    model = Pipeline([("scale", StandardScaler()),
                      ("model", RidgeCV(alphas=np.logspace(-3, 5, 20)))])
    model.fit(X[train_rows], y[train_rows])
    predicted = model.predict(X[test_rows])
    return predicted, train.score_rows("regression", y[test_rows], predicted)


representations = {"Composition (GC, GC3, codon use, length)": composition, **pooled,
                   "BLF three encoders concatenated": np.hstack(list(pooled.values()))}
if streams:
    representations["LucaOne, gene stream"] = streams["gene"]
    representations["LucaOne, protein stream"] = streams["prot"]
    representations["LucaOne, both streams"] = np.hstack([streams["gene"], streams["prot"]])

scores, predictions = {}, {}
for name, X in representations.items():
    predictions[name], scores[name] = probe_on_split(X, y, train_rows, test_rows)
    print(f"  {name}: Spearman {scores[name]:.4f}")

summary = pd.DataFrame({"probe Spearman": pd.Series(scores),
                        "width": {k: v.shape[1] for k, v in representations.items()}})
display(summary.sort_values("probe Spearman", ascending=False).round(4))

### Does the jointly pretrained model beat the separately pretrained ones?

The same paired bootstrap the rest of the project uses: the test rows are resampled, every
model is scored on the same resample, and the interval is over their difference. Sequences
that repeat in the test split are resampled together, because the stability split repeats
sequences and counting them separately would overstate the evidence.

In [ ]:
if "LucaOne, both streams" in predictions:
    comparisons = [
        ("LucaOne both streams minus the best BLF single encoder",
         "LucaOne, both streams", max(pooled, key=lambda k: scores[k])),
        ("LucaOne both streams minus the three concatenated",
         "LucaOne, both streams", "BLF three encoders concatenated"),
        ("LucaOne gene stream minus composition",
         "LucaOne, gene stream", "Composition (GC, GC3, codon use, length)"),
    ]
    found = train.seed_averaged_bootstrap(
        "regression", y[test_rows], {k: [v] for k, v in predictions.items()}, comparisons,
        groups=rows["sequence"].to_numpy()[test_rows], draws=BOOTSTRAP_DRAWS)
    display(pd.DataFrame(found).set_index("comparison").round(4))
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(found).to_csv(RESULTS_DIR / f"{BENCHMARK}_comparisons.csv", index=False)
    summary.to_csv(RESULTS_DIR / f"{BENCHMARK}_probe_scores.csv")
else:
    print("Run the LucaOne cells first; this cell compares what they produce.")

### Beside the trained head

The probe table above is a representation comparison. Phase A's numbers come from the
trained TextCNN on the token grid, a different and stronger head, and are printed here only
so the two are not read as the same measurement.

In [ ]:
phase_a = []
for path in sorted((Path("results") / "phaseA" / BENCHMARK).glob("*.json")):
    result = train.Result.load(path)
    phase_a.append({"variant": result.variant, "seed": result.config["seed"],
                    "test": result.test_score})
if phase_a:
    best = pd.DataFrame(phase_a).groupby("variant")["test"].mean().sort_values(ascending=False)
    display(best.round(4).to_frame("Phase A, trained TextCNN"))
    print("Different head, same rows and same split: not a matched comparison with the probe "
          "table, which is why it is in its own table.")
else:
    print("No Phase A results found for this benchmark yet.")

## Part 4: IsoFormer's GTEx table at three seeds (Gantt 44)

Stage 3 step 4 ran the six-row modality ablation at seed 0 and found that fusion beats the
best single modality by $+0.047$. One seed cannot say how much of that is the split and how
much is initialisation, and IsoFormer itself reports five seeds with a standard deviation on
every number. Seeds 1 and 2 are run here, into their own results folder, and the three are
read together.

This is the one expensive cell in this notebook that does not involve LucaOne; on a CUDA GPU
it is roughly an hour, on the Mac about four.

In [ ]:
RUN_ISOFORMER_SEEDS = False          # turn on when a GPU night is available
ISOFORMER_SEEDS = (1, 2)
VARIANTS = ("dna", "rna", "prot", "dna+prot", "dna+rna", "dna+rna+prot")
MAX_BATCH, CELL_BUDGET, N_HEADS, VAL_FRACTION, MAX_EPOCHS = 16, 300_000_000, 8, 0.05, 500

if RUN_ISOFORMER_SEEDS:
    gtex_streams = isoformer.GTExStreams.open(CACHE_ROOT)
    for seed in ISOFORMER_SEEDS:
        print(f"\n--- seed {seed} ---")
        for variant in VARIANTS:
            made = isoformer.loaders_for(gtex_streams, batch_size=MAX_BATCH,
                                         val_fraction=VAL_FRACTION, seed=seed,
                                         budget=CELL_BUDGET, n_heads=N_HEADS)
            train.run(gtex_streams, variant,
                      train.TrainConfig(batch_size=MAX_BATCH, max_epochs=MAX_EPOCHS, seed=seed),
                      DEVICE, GTEX_RESULTS, builder=isoformer.builder, loaders=made,
                      tag=f"gtex_pilot__{variant}__heads{N_HEADS}"
                          f"__budget{CELL_BUDGET // 1_000_000}M__seed{seed}",
                      n_heads=N_HEADS)
else:
    print("RUN_ISOFORMER_SEEDS is off; the reading cell below works with whatever seeds exist.")

In [ ]:
# Seed 0 lives with Stage 3 step 4; seeds 1 and 2 land here. Both are read.
ISOFORMER_TABLE2 = pd.DataFrame(
    [("dna", 0.43), ("rna", 0.61), ("prot", 0.46),
     ("dna+prot", 0.52), ("dna+rna", 0.64), ("dna+rna+prot", 0.65)],
    columns=["variant", "their Spearman"]).set_index("variant")

found = []
for folder in (Path("../stage3/results/gtex_pilot"), GTEX_RESULTS):
    for path in sorted(folder.glob("*.json")) if folder.exists() else []:
        result = train.Result.load(path)
        found.append({"variant": result.variant, "seed": result.config["seed"],
                      "Spearman": result.test_score})
if found:
    runs = pd.DataFrame(found).drop_duplicates(["variant", "seed"])
    table = runs.groupby("variant")["Spearman"].agg(["mean", "std", "count"])
    display(table.join(ISOFORMER_TABLE2).round(3))
    if table["count"].max() > 1:
        print(f"seed spread, largest standard deviation: {table['std'].max():.3f}")
else:
    print("No IsoFormer runs found yet.")

## Deviations recorded for this notebook

| # | deviation | why |
| --- | --- | --- |
| 1 | **LucaOne is read as a frozen encoder, pooled to one vector, and probed with ridge** | the plan asks for a frozen baseline with "the same train/test folds and prediction head or clearly matched probe setting"; LucaOne returns no token grid this project's TextCNN could read, so the matched setting is the probe, applied identically to every representation in the table |
| 2 | **No LucaOne baseline on GTEx** | its 1,280-token context admits none of the pilot's 6,000-nucleotide DNA windows; Part 1 measures this rather than assuming it |
| 3 | The published 1,280-token limit is used, although the checkpoint reports 4,096 positions with position embeddings disabled | the paper supports 1,280; anything longer would be an extrapolation this project has not tested |
| 4 | The joint gene-and-protein input is not run | it caps the coding sequence at 957 nucleotides, which drops part of the benchmark, so the two streams are embedded separately and concatenated, and that is what "both streams" means in the table |
| 5 | float16 on CUDA, float32 on CPU and Apple GPU | 1.8B parameters in float32 do not fit an 8 GB card beside their activations |
| 6 | BioLangFusion's encoders are pooled from the Stage 3 token caches rather than re-embedded | same weights, same rows, same pooling, and no second extraction; the pooling is the mean over content tokens, as in Stage 1 |
| 7 | IsoFormer seeds 1 and 2 only, against its five | the pilot is a 2,000-row sample and each seed is an hour of GPU time; three seeds give a spread, which is what the one-seed table lacked |